In [ ]:
#!/usr/bin/env python3
"""
detect_spacegroup.py

读取结构文件（cif / POSCAR / CONTCAR / xyz / cif-like），输出：
- space group number
- space group symbol
- crystal system
- 每个原子对应的 Wyckoff position（按输入顺序），以及归并后的 Wyckoff site 列表（唯一位置、乘冗、占据元素、代表坐标）

依赖: pymatgen, spglib
pip install pymatgen spglib
"""

import sys
import argparse
from collections import defaultdict

from pymatgen.core import Structure, Lattice, Molecule
from pymatgen.io.vasp import Poscar
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer


def load_structure(filename, default_box=20.0):
    """
    尝试自动加载结构文件。支持 cif / POSCAR / CONTCAR / xyz 等。
    如果是没有晶格信息的分子（pymatgen 返回 Molecule），会将其放入默认立方晶格。
    """
    try:
        s = Structure.from_file(filename)  # pymatgen 会智能判断多数格式
        # 如果 returned object is Molecule, try convert
        if isinstance(s, Molecule):
            # convert to Structure by placing into a cubic lattice
            lat = Lattice.cubic(default_box)
            coords = s.cart_coords
            species = [site.species_string for site in s.sites]
            struct = Structure(lat, species, coords, coords_are_cartesian=True)
            return struct, False  # False 表示晶格是人为添加的
        return s, True
    except Exception as e:
        # 尝试特别处理 POSCAR（有时文件名不同）
        try:
            pos = Poscar.from_file(filename)
            return pos.structure, True
        except Exception:
            raise RuntimeError(f"无法读取文件 '{filename}'：{e}")


def analyze_structure(structure, symprec=1e-3):
    sga = SpacegroupAnalyzer(structure, symprec=symprec)
    sg_symbol = sga.get_space_group_symbol()
    sg_number = sga.get_space_group_number()
    crystal_system = sga.get_crystal_system()
    dataset = sga.get_symmetry_dataset()
    # dataset 包含 keys: 'equivalent_atoms', 'wyckoffs', 'rotations', 'translations', ...
    wyckoffs = dataset.get("wyckoffs", None)
    equivalent_atoms = dataset.get("equivalent_atoms", None)

    # per-atom wyckoff (same order as structure.sites)
    per_atom = []
    if wyckoffs is not None:
        for i, site in enumerate(structure.sites):
            per_atom.append({
                "index": i,
                "species": site.species_string,
                "frac_coords": tuple(site.frac_coords),
                "wyckoff": wyckoffs[i]
            })
    else:
        # fallback: empty
        for i, site in enumerate(structure.sites):
            per_atom.append({
                "index": i,
                "species": site.species_string,
                "frac_coords": tuple(site.frac_coords),
                "wyckoff": None
            })

    # aggregate unique Wyckoff sites by equivalent_atoms index (if provided)
    unique_sites = []
    if equivalent_atoms is not None and wyckoffs is not None:
        groups = defaultdict(list)
        for i, eq in enumerate(equivalent_atoms):
            groups[eq].append(i)
        for eq, indices in sorted(groups.items()):
            rep = indices[0]
            multiplicity = len(indices)
            w = wyckoffs[rep]
            species_set = sorted({structure[i].species_string for i in indices})
            frac = tuple(structure[rep].frac_coords)
            unique_sites.append({
                "representative_index": rep,
                "wyckoff": w,
                "multiplicity": multiplicity,
                "species": species_set,
                "frac_coords": frac,
                "indices": indices
            })
    else:
        # fallback: list each site individually
        for i, site in enumerate(structure.sites):
            unique_sites.append({
                "representative_index": i,
                "wyckoff": wyckoffs[i] if wyckoffs is not None else None,
                "multiplicity": 1,
                "species": [site.species_string],
                "frac_coords": tuple(site.frac_coords),
                "indices": [i]
            })

    return {
        "spacegroup_symbol": sg_symbol,
        "spacegroup_number": sg_number,
        "crystal_system": crystal_system,
        "per_atom": per_atom,
        "unique_sites": unique_sites,
        "symprec": symprec
    }


def pretty_print(result, structure, lattice_was_real=True, filename=None):
    header = f"File: {filename}" if filename else "Structure"
    print("=" * 60)
    print(header)
    print("=" * 60)
    print(f"Space group: {result['spacegroup_symbol']}  (No. {result['spacegroup_number']})")
    print(f"Crystal system: {result['crystal_system']}")
    print(f"Symmetry precision (symprec): {result['symprec']}")
    if not lattice_was_real:
        print("\n注意：输入结构没有晶格信息，脚本已用默认立方盒子替代。"
              "在没有真实晶格的情况下，空间群判定可能没有物理意义。")
    print("\nLattice vectors (fractional basis):")
    a, b, c = structure.lattice.matrix
    print(f" a: {a}")
    print(f" b: {b}")
    print(f" c: {c}")
    print("\nPer-atom Wyckoff positions (in input order):")
    print(f"{'idx':>4} {'species':>8} {'wyckoff':>8} {'frac coords':>30}")
    for p in result["per_atom"]:
        wy = p["wyckoff"] if p["wyckoff"] is not None else "-"
        coords = "({:.4f}, {:.4f}, {:.4f})".format(*p["frac_coords"])
        print(f"{p['index']:4d} {p['species']:>8} {wy:>8} {coords:>30}")
    print("\nUnique Wyckoff sites (merged equivalent atoms):")
    print(f"{'rep':>4} {'wyckoff':>8} {'mult':>4} {'species':>12} {'frac coords':>30} {'indices':>12}")
    for u in result["unique_sites"]:
        wy = u["wyckoff"] if u["wyckoff"] is not None else "-"
        species = ",".join(u["species"])
        coords = "({:.4f}, {:.4f}, {:.4f})".format(*u["frac_coords"])
        inds = ",".join(map(str, u["indices"]))
        print(f"{u['representative_index']:4d} {wy:>8} {u['multiplicity']:4d} {species:>12} {coords:>30} {inds:>12}")
    print("=" * 60)


def main():
    parser = argparse.ArgumentParser(description="Detect space group and Wyckoff positions from a structure file.")
    parser.add_argument("file", help="Structure file (CIF, POSCAR/CONTCAR, xyz, etc.)")
    parser.add_argument("--symprec", type=float, default=1e-3, help="tolerance for symmetry finding (default: 1e-3)")
    parser.add_argument("--default-box", type=float, default=20.0, help="for molecule/xyz without cell, cubic box size (Å)")
    args = parser.parse_args()

    try:
        struct, has_lattice = load_structure(args.file, default_box=args.default_box)
    except Exception as e:
        print(f"Error loading structure: {e}", file=sys.stderr)
        sys.exit(2)

    result = analyze_structure(struct, symprec=args.symprec)
    pretty_print(result, struct, lattice_was_real=has_lattice, filename=args.file)


if __name__ == "__main__":
    main()


: 